# Random Forest from scratch 
Bootstrap sampling -> feature subsampling -> majority vote -> OOB score ->
MDI vs permutation importance -> effect of n_trees and max_features -> Play Tennis example.
A compact CART decision tree is included so this file is self-contained.

In [ ]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.makedirs('plots', exist_ok=True)
rng = np.random.default_rng(0)

## 1. A compact decision tree (see 07_Decision_Trees for the full version)
Supports gini/entropy, max_depth, min_samples_split and `max_features` (random feature
subset examined at *every split* - the key ingredient of a random forest).

In [ ]:
def gini_impurity(y, k):
    p = np.bincount(y, minlength=k) / len(y)
    return 1.0 - np.sum(p ** 2)

def entropy(y, k):
    p = np.bincount(y, minlength=k) / len(y)
    p = p[p > 0]
    return -np.sum(p * np.log2(p))

class DecisionTree:
    def __init__(self, max_depth=None, min_samples_split=2, max_features=None,
                 criterion="gini", rng=None):
        self.max_depth, self.min_samples_split = max_depth, min_samples_split
        self.max_features, self.criterion = max_features, criterion
        self.rng = rng or np.random.default_rng(0)

    def _imp(self, y):
        return (gini_impurity if self.criterion == "gini" else entropy)(y, self.k)

    def fit(self, X, y):
        self.k = int(y.max()) + 1
        self.d = X.shape[1]
        self.feature_importances_ = np.zeros(self.d)
        self.tree = self._build(X, y, 0)
        s = self.feature_importances_.sum()
        if s > 0: self.feature_importances_ /= s
        return self

    def _best_split(self, X, y):
        n, d = X.shape
        m = d if self.max_features is None else self.max_features
        feats = self.rng.choice(d, m, replace=False)
        parent = self._imp(y)
        best = (None, None, 0.0)
        for j in feats:
            order = np.argsort(X[:, j], kind="stable")
            xs, ys = X[order, j], y[order]
            oh = np.eye(self.k)[ys]
            cl = np.cumsum(oh, 0)[:-1]; cr = oh.sum(0) - cl
            nl = np.arange(1, n)[:, None]; nr = n - nl
            pl, pr = cl / nl, cr / nr
            if self.criterion == "gini":
                il, ir = 1 - (pl**2).sum(1), 1 - (pr**2).sum(1)
            else:
                il = -np.where(pl > 0, pl * np.log2(np.where(pl > 0, pl, 1)), 0).sum(1)
                ir = -np.where(pr > 0, pr * np.log2(np.where(pr > 0, pr, 1)), 0).sum(1)
            gain = parent - (nl[:, 0] / n) * il - (nr[:, 0] / n) * ir
            gain = np.where(xs[1:] != xs[:-1], gain, -np.inf)
            i = int(np.argmax(gain))
            if gain[i] > best[2] + 1e-12:
                best = (int(j), (xs[i] + xs[i+1]) / 2, float(gain[i]))
        return best

    def _build(self, X, y, depth):
        n = len(y)
        leaf = ("leaf", np.bincount(y, minlength=self.k) / n)   # class probabilities
        if (len(np.unique(y)) == 1 or n < self.min_samples_split or
                (self.max_depth is not None and depth >= self.max_depth)):
            return leaf
        f, t, g = self._best_split(X, y)
        if f is None:
            return leaf
        self.feature_importances_[f] += n * g
        mask = X[:, f] <= t
        return ("split", f, t, self._build(X[mask], y[mask], depth+1), self._build(X[~mask], y[~mask], depth+1))

    def predict_proba(self, X):
        out = np.zeros((len(X), self.k))
        for i, x in enumerate(X):
            node = self.tree
            while node[0] == "split":
                node = node[3] if x[node[1]] <= node[2] else node[4]
            out[i] = node[1]
        return out

    def predict(self, X):
        return self.predict_proba(X).argmax(1)

## 2. Bootstrap sampling
Draw n indices with replacement. On average ~63.2% of rows appear; ~36.8% are out-of-bag (OOB).

In [ ]:
def bootstrap_sample(X, y, rng):
    n = len(X)
    idx = rng.integers(0, n, n)
    return X[idx], y[idx], idx

n = 10000
idx = rng.integers(0, n, n)
print("fraction unique in bootstrap: %.3f (theory 1-1/e = %.3f)" % (len(np.unique(idx)) / n, 1 - 1 / np.e))

## 3. RandomForest class
fit: for each tree, bootstrap + tree with random feature subset per split (max_features='sqrt').
predict: majority vote (we also expose averaged probabilities).
OOB score: each sample is voted on only by trees that did not see it.

In [ ]:
class RandomForest:
    def __init__(self, n_trees=100, max_depth=None, min_samples_split=2,
                 max_features="sqrt", criterion="gini", seed=0):
        self.n_trees, self.max_depth = n_trees, max_depth
        self.min_samples_split, self.max_features = min_samples_split, max_features
        self.criterion, self.seed = criterion, seed

    def _mf(self, d):
        if self.max_features == "sqrt": return max(1, int(np.sqrt(d)))
        if self.max_features == "third": return max(1, d // 3)
        if self.max_features is None: return d
        return int(self.max_features)

    def fit(self, X, y):
        rng = np.random.default_rng(self.seed)
        n, d = X.shape
        self.k = int(y.max()) + 1
        self.trees, self.oob_masks = [], []
        oob_votes = np.zeros((n, self.k))
        for _ in range(self.n_trees):
            Xb, yb, idx = bootstrap_sample(X, y, rng)
            tree = DecisionTree(self.max_depth, self.min_samples_split, self._mf(d),
                                self.criterion, rng=np.random.default_rng(rng.integers(1 << 31)))
            tree.fit(Xb, yb)
            self.trees.append(tree)
            oob = np.ones(n, bool); oob[idx] = False
            self.oob_masks.append(oob)
            if oob.any():
                oob_votes[oob] += tree.predict_proba(X[oob])
        has = oob_votes.sum(1) > 0
        self.oob_score_ = float(np.mean(oob_votes[has].argmax(1) == y[has]))
        self.feature_importances_ = forest_feature_importances(self)
        return self

    def predict_proba(self, X):
        return np.mean([t.predict_proba(X) for t in self.trees], axis=0)

    def predict_votes(self, X):
        """Hard majority vote (mode of tree predictions)."""
        votes = np.zeros((len(X), self.k), int)
        for t in self.trees:
            votes[np.arange(len(X)), t.predict(X)] += 1
        return votes.argmax(1)

    def predict(self, X):
        return self.predict_votes(X)

def forest_feature_importances(forest):
    """Average of per-tree normalised MDI importances."""
    return np.mean([t.feature_importances_ for t in forest.trees], axis=0)

def permutation_importance(model, X, y, n_repeats=5, seed=0):
    """Accuracy drop when one column is shuffled (computed on held-out data)."""
    r = np.random.default_rng(seed)
    base = np.mean(model.predict(X) == y)
    imp = np.zeros(X.shape[1])
    for j in range(X.shape[1]):
        drops = []
        for _ in range(n_repeats):
            Xp = X.copy(); Xp[:, j] = r.permutation(Xp[:, j])
            drops.append(base - np.mean(model.predict(Xp) == y))
        imp[j] = np.mean(drops)
    return imp

def split(X, y, frac=0.3, seed=0):
    r = np.random.default_rng(seed); p = r.permutation(len(y)); k = int(len(y) * frac)
    return X[p[k:]], X[p[:k]], y[p[k:]], y[p[:k]]

## 4. Synthetic dataset: 2 informative features, 2 noisy, one high-cardinality noise, one binary noise

In [ ]:
N = 500
X1 = rng.normal(size=(N, 2))
y_all = ((X1[:, 0] * X1[:, 1] > 0) ^ (rng.random(N) < 0.08)).astype(int)     # XOR-like -> nonlinear
noise = rng.normal(size=(N, 2))
hc_noise = rng.integers(0, 200, (N, 1)).astype(float)                       # high-cardinality noise
X_all = np.hstack([X1, noise, hc_noise])
fnames = ["informative_1", "informative_2", "noise_1", "noise_2", "hi_card_noise"]
Xtr, Xte, ytr, yte = split(X_all, y_all)

t = DecisionTree(rng=np.random.default_rng(1)).fit(Xtr, ytr)
rf = RandomForest(n_trees=60, seed=1).fit(Xtr, ytr)
print(f"single tree test acc = {np.mean(t.predict(Xte)==yte):.3f}")
print(f"forest  test acc = {np.mean(rf.predict(Xte)==yte):.3f}   OOB score = {rf.oob_score_:.3f}")

## 5. Number of trees (exercise 1) and OOB vs test estimate (exercise 4)

In [ ]:
counts = [1, 5, 10, 25, 50, 100]
tr_a, te_a, oob_a = [], [], []
for m in counts:
    f = RandomForest(n_trees=m, seed=2).fit(Xtr, ytr)
    tr_a.append(np.mean(f.predict(Xtr) == ytr)); te_a.append(np.mean(f.predict(Xte) == yte)); oob_a.append(f.oob_score_)
    print(f"trees={m:3d} train={tr_a[-1]:.3f} test={te_a[-1]:.3f} oob={oob_a[-1]:.3f}")
fig, ax = plt.subplots(figsize=(5.5, 3.6))
ax.plot(counts, tr_a, 'o-', label='train'); ax.plot(counts, te_a, 's-', label='test'); ax.plot(counts, oob_a, '^--', label='OOB')
ax.set_xscale('log'); ax.set_xlabel('n_trees'); ax.set_ylabel('accuracy'); ax.legend()
fig.tight_layout(); fig.savefig('plots/01_n_trees.png', dpi=120); plt.close(fig)

## 6. MDI vs permutation importance (exercise 2)
MDI tends to inflate the high-cardinality noise feature (many candidate thresholds -> lucky splits).

In [ ]:
mdi = rf.feature_importances_
perm = permutation_importance(rf, Xte, yte)
print("MDI :", dict(zip(fnames, np.round(mdi, 3))))
print("Perm:", dict(zip(fnames, np.round(perm, 3))))
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
axs[0].barh(fnames, mdi); axs[0].set_title('MDI importance')
axs[1].barh(fnames, perm); axs[1].set_title('Permutation importance (test)')
fig.tight_layout(); fig.savefig('plots/02_importance.png', dpi=120); plt.close(fig)

## 7. max_features: tree correlation vs strength (exercise 3)
Fewer features per split -> weaker but less correlated trees. We measure the mean pairwise
correlation of tree predictions on the test set.

In [ ]:
def mean_tree_corr(forest, X):
    P = np.array([t.predict(X) for t in forest.trees]).astype(float)
    P = P[P.std(1) > 0]
    C = np.corrcoef(P)
    return float(C[np.triu_indices(len(P), 1)].mean())

res = {}
for mf in [1, 2, "sqrt", 5]:
    f = RandomForest(n_trees=40, max_features=mf, seed=3).fit(Xtr, ytr)
    tree_acc = np.mean([np.mean(t.predict(Xte) == yte) for t in f.trees])
    res[str(mf)] = (tree_acc, np.mean(f.predict(Xte) == yte), mean_tree_corr(f, Xte))
    print(f"max_features={mf!s:5} mean tree acc={res[str(mf)][0]:.3f} forest acc={res[str(mf)][1]:.3f} tree corr={res[str(mf)][2]:.3f}")
fig, ax = plt.subplots(figsize=(5.5, 3.6))
ks = list(res)
ax.plot(ks, [res[k][0] for k in ks], 'o-', label='avg single tree'); ax.plot(ks, [res[k][1] for k in ks], 's-', label='forest')
ax.plot(ks, [res[k][2] for k in ks], '^--', label='tree correlation')
ax.set_xlabel('max_features'); ax.legend(); fig.tight_layout(); fig.savefig('plots/03_max_features.png', dpi=120); plt.close(fig)

## 8. Decision boundary: one deep tree vs forest (variance reduction)

In [ ]:
Xb2 = rng.normal(size=(300, 2)); yb2 = ((Xb2[:, 0]**2 + Xb2[:, 1]**2 + rng.normal(0, 0.6, 300)) > 1.5).astype(int)
gx, gy = np.meshgrid(np.linspace(-3, 3, 100), np.linspace(-3, 3, 100)); G = np.c_[gx.ravel(), gy.ravel()]
tree2 = DecisionTree(rng=np.random.default_rng(0)).fit(Xb2, yb2)
rf2 = RandomForest(n_trees=50, max_features=1, seed=0).fit(Xb2, yb2)
fig, axs = plt.subplots(1, 2, figsize=(9, 4))
for ax, m, ttl in zip(axs, [tree2, rf2], ["single tree", "random forest (50)"]):
    ax.contourf(gx, gy, m.predict(G).reshape(gx.shape), alpha=0.3, cmap='coolwarm')
    ax.scatter(Xb2[:, 0], Xb2[:, 1], c=yb2, cmap='coolwarm', s=8, edgecolor='k', linewidth=0.2); ax.set_title(ttl)
fig.tight_layout(); fig.savefig('plots/04_boundary.png', dpi=120); plt.close(fig)

## 9. Play Tennis worked example (Math/math.md)
One-hot encode the categorical columns so our numeric tree can split on them (m = sqrt(p) features per split).

In [ ]:
rows = """Sunny Hot High False No
Sunny Hot High True No
Overcast Hot High False Yes
Rain Mild High False Yes
Rain Cool Normal False Yes
Rain Cool Normal True No
Overcast Cool Normal True Yes
Sunny Mild High False No
Sunny Cool Normal False Yes
Rain Mild Normal False Yes
Sunny Mild Normal True Yes
Overcast Mild High True Yes
Overcast Hot Normal False Yes
Rain Mild High True No""".split("\n")
data = [r.split() for r in rows]
cols = ["Outlook", "Temperature", "Humidity", "Windy"]
levels = [sorted(set(r[j] for r in data)) for j in range(4)]
onehot_names = [f"{c}={v}" for c, lv in zip(cols, levels) for v in lv]
def encode(r):
    return [1.0 if r[j] == v else 0.0 for j in range(4) for v in levels[j]]
XT = np.array([encode(r) for r in data]); yT = np.array([1 if r[4] == "Yes" else 0 for r in data])
py = yT.mean()
print(f"Root entropy = {-(py*np.log2(py)+(1-py)*np.log2(1-py)):.3f}")
rfT = RandomForest(n_trees=3, max_features=2, criterion="entropy", seed=4).fit(XT, yT)
Q = np.array([encode(["Sunny", "Cool", "High", "False"])])
print("individual tree votes for Q:", [("Yes" if t.predict(Q)[0] else "No") for t in rfT.trees])
print("3-tree forest prediction for Q:", "Yes" if rfT.predict(Q)[0] else "No", "(doc: No)")
rfT = RandomForest(n_trees=100, max_features=2, criterion="entropy", seed=4).fit(XT, yT)
print("100-tree forest prediction for Q:", "Yes" if rfT.predict(Q)[0] else "No", "| OOB score:", round(rfT.oob_score_, 3))
top = np.argsort(rfT.feature_importances_)[::-1][:4]
print("top MDI features:", [(onehot_names[i], round(rfT.feature_importances_[i], 3)) for i in top])